# Notebook 3 — Fine-tuning con LoRA: Qwen3-4B

Este notebook realiza el fine-tuning del modelo **Qwen3-4B** mediante **LoRA** (Low-Rank Adaptation) para la tarea de clasificacion ideologica de noticias espanolas en tres clases:
**IZQUIERDA**, **CENTRO** y **DERECHA**.

### Variante: LoRA r=32, lora_alpha=64, target_modules MLP (gate/up/down_proj)

### Justificacion de LoRA
Se usa LoRA en lugar de fine-tuning completo por tres razones:
1. **Pocos datos**: 1,902 ejemplos de entrenamiento son insuficientes para reentrenar 4B parametros sin sobreajuste. LoRA entrena solo ~1% de parametros adicionales (Hu et al., 2021).
2. **VRAM limitada**: Qwen3-4B en full fine-tuning requiere >40 GB de VRAM. Con LoRA mas cuantizacion 4-bit cabe en ~9 GB (T4 de Colab).
3. **Preservar conocimiento**: LoRA congela los pesos del modelo base, preservando su comprension del espanol y adaptando solo las capas de atencion al dominio ideologico.

Hu, E. J. et al. (2021). LoRA: Low-Rank Adaptation of Large Language Models. arXiv:2106.09685.


## 1. Montar Google Drive


In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

assert os.path.exists('/content/drive/MyDrive'), (
    'Drive no montado correctamente. Vuelve a ejecutar esta celda.'
)
print('Drive montado correctamente')
print('Contenido de MyDrive/TFM (si existe):')
tfm_path = '/content/drive/MyDrive/TFM'
if os.path.exists(tfm_path):
    for item in sorted(os.listdir(tfm_path)):
        print(f'  {item}')
else:
    print('  La carpeta TFM no existe todavia en Drive. Se creara al guardar.')


Mounted at /content/drive
Drive montado correctamente
Contenido de MyDrive/TFM (si existe):
  qwen3_4b_lora
  qwen3_4b_lora_mlp16
  qwen3_4b_lora_r8_qkv
  qwen3_4b_lora_regularized
  qwen3_4b_qlora
  splits


## 2. Instalación de dependencias e Imports

Se instalan todas las dependencias y se centralizan todos los imports de Python en las celdas de esta sección.

In [2]:
# Verificar GPU antes de instalar
!nvidia-smi

!pip install -q --upgrade transformers>=4.51.0 peft>=0.14.0 \
             bitsandbytes accelerate datasets sentencepiece trl
!pip install -q --upgrade torchao
!pip install -q lime

Tue May 19 17:31:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 3. Imports y configuración global

Todos los imports de Python están en la celda siguiente. A continuación se definen las constantes de configuración: SEED, codificación de etiquetas y rutas de datos y modelos.

In [3]:
# (pip install movido a la celda de dependencias al inicio)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [4]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_recall_fscore_support,
    cohen_kappa_score, confusion_matrix,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd



import shap
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd


import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import log_softmax

from lime.lime_text import LimeTextExplainer
import matplotlib.patches as mpatches
import torch
from sklearn.metrics import f1_score
from IPython.display import display, HTML
import transformers
print(f'transformers version: {transformers.__version__}')

In [5]:
# Semilla global para reproducibilidad (permite comparar notebooks)
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Codificacion de etiquetas
LABEL2ID = {'IZQUIERDA': 0, 'CENTRO': 1, 'DERECHA': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
CLASES   = ['IZQUIERDA', 'CENTRO', 'DERECHA']

MODEL_NAME = 'Qwen/Qwen3-4B'
OUTPUT_DIR = '/content/drive/MyDrive/TFM/qwen3_4b_lora_mlp32'

# Copiar datos de Drive a disco local para mayor velocidad
import shutil, os
os.makedirs('/content/splits', exist_ok=True)
shutil.copy('/content/drive/MyDrive/TFM/splits/df_train.parquet', '/content/splits/df_train.parquet')
shutil.copy('/content/drive/MyDrive/TFM/splits/df_val.parquet',   '/content/splits/df_val.parquet')
SPLITS_DIR = '/content/splits/'

# Crear carpeta de output si no existe
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Verificar GPU
print(f'GPU disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'SPLITS_DIR     : {SPLITS_DIR}')
print(f'OUTPUT_DIR     : {OUTPUT_DIR}')


GPU disponible : True
GPU            : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM total     : 102.0 GB
SPLITS_DIR     : /content/splits/
OUTPUT_DIR     : /content/drive/MyDrive/TFM/qwen3_4b_lora_mlp32


## 4. Cargar datos de entrenamiento y validacion

Cargamos df_train y df_val desde Drive. La columna `texto` contiene el articulo
completo (titular + abstract + noticia) construido en el notebook de preprocesado.
La columna `final_label` contiene la etiqueta ideologica del ground truth.

df_test NO se carga aqui. Se reserva para la comparativa final de la Seccion 6, ya que si se usara aqui para ajustar hiperparametros dejaria de ser una medida honesta del modelo.

In [6]:
df_train = pd.read_parquet(SPLITS_DIR + 'df_train.parquet')
df_val   = pd.read_parquet(SPLITS_DIR + 'df_val.parquet')

# Verificar columnas
print('Columnas df_train:', df_train.columns.tolist())
assert 'texto' in df_train.columns, 'Columna texto no encontrada en df_train'
assert 'final_label' in df_train.columns, 'Columna final_label no encontrada en df_train'

print(f'\nTrain : {len(df_train):,} articulos')
print(f'Val   : {len(df_val):,} articulos')
print(f'\nDistribucion train:')
print(df_train['final_label'].value_counts().to_string())
print(f'\nDistribucion val:')
print(df_val['final_label'].value_counts().to_string())
print(f'\nLongitud media texto train : {df_train["texto"].str.len().mean():.0f} caracteres')
print(f'Longitud maxima texto train: {df_train["texto"].str.len().max():,} caracteres')


Columnas df_train: ['id', 'texto', 'medio', 'final_label', 'accion']

Train : 1,902 articulos
Val   : 238 articulos

Distribucion train:
final_label
CENTRO       766
IZQUIERDA    672
DERECHA      464

Distribucion val:
final_label
CENTRO       96
IZQUIERDA    84
DERECHA      58

Longitud media texto train : 4521 caracteres
Longitud maxima texto train: 32,165 caracteres


## 5. Cargar el modelo

Se usa AutoModelForSequenceClassification en lugar del modelo generativo porque:
- Añade una cabeza de clasificacion lineal sobre el transformer
- Es mas eficiente para clasificacion que generar texto token a token
- Permite obtener probabilidades directamente via softmax sobre los logits

In [7]:
# Sin cuantizacion — carga estandar bfloat16
print('Cargando tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Vocabulario: {tokenizer.vocab_size:,} tokens')

print('\nCargando modelo (1-2 min)...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

vram_used = torch.cuda.memory_allocated() / 1e9
print(f'Modelo cargado VRAM usada: {vram_used:.1f} GB')


Cargando tokenizer...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-4B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado VRAM usada: 8.0 GB


## 6. Configurar LoRA

**Variante: LoRA MLP r32**

Se adaptan las proyecciones MLP del transformer (gate_proj, up_proj, down_proj) con rango r=32 y lora_alpha=64.

Parámetros clave:
- `r=32`: rango alto — mayor capacidad para capturar patrones complejos en las capas feed-forward, pero con mayor riesgo de sobreajuste dado el tamaño reducido del dataset (1,902 ejemplos de entrenamiento).
- `lora_alpha=64`: escala de LoRA. Regla habitual: alpha = 2 × r.
- `lora_dropout=0.1`: regularización para limitar la memorización del conjunto de entrenamiento.
- `target_modules=['gate_proj', 'up_proj', 'down_proj']`: se adaptan las tres proyecciones de las capas feed-forward, que almacenan el conocimiento factual del modelo. La hipótesis es que estas capas pueden contener información relevante para el sesgo ideológico léxico.

Diferencia respecto a las variantes de atención (QV, QKV): en lugar de adaptar cómo el modelo atiende a distintas posiciones del texto, aquí se adapta cómo procesa la información dentro de cada posición.

In [8]:
# (pip install movido a la celda de dependencias al inicio)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.7 MB/s eta 0:00:00


In [9]:
# Notebook 3: r=32, alpha=64, targets MLP (gate/up/down_proj)
# Logica: igual que NB2 pero con mayor rango -> mas capacidad para capturar
# patrones ideologicos en las capas MLP del transformer.
# Comparar vs NB2 (r=16 MLP) para ver si el rango extra ayuda o aumenta overfitting.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=['gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 42,475,008 || all params: 4,064,950,784 || trainable%: 1.0449


### Parámetros entrenables

Con r=32 sobre gate_proj, up_proj y down_proj se entrenan **42,475,008 parámetros** (1.0449 % del total de Qwen3-4B). Es la variante con mayor número de parámetros entrenables de toda la batería, aproximadamente 7 veces más que las variantes de atención (QKV: 4.0 M, QV: 5.9 M).

Un mayor número de parámetros entrenables no implica necesariamente un mejor rendimiento con datasets pequeños. Con solo 1,902 ejemplos de entrenamiento, el riesgo de memorización aumenta proporcionalmente, como confirma el diagnóstico de overfitting BASTANTE observado en las curvas de aprendizaje.

## 7. Tokenizar los datasets

Convertimos los DataFrames de pandas a formato HuggingFace Dataset.

El tokenizer convierte el texto en secuencias de token IDs que el modelo entiende.
Se aplica truncado a max_length=1024 tokens cuando el articulo es mas largo.
No se trunca el texto manualmente antes: se pasa el campo texto completo
y el tokenizer decide el corte.

Con una longitud media de ~ 4,455 caracteres (~1,100 tokens), max_length=1024 cubre aproximadamente el 70 por ciento de los articulos completos.
La informacion ideologica relevante se concentra en el titular y los primeros parrafos, por lo que el truncado no supone una perdida significativa de senal.

padding=False en el tokenizer y DataCollatorWithPadding en el Trainer hacen padding
dinamico por batch: cada batch se padea solo hasta el ejemplo mas largo de ese batch,
no hasta max_length. Esto ahorra VRAM y acelera el entrenamiento.


In [10]:
MAX_LENGTH = 1024

def tokenize(batch):
    return tokenizer(
        batch['texto'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

def make_dataset(df):
    ds = Dataset.from_dict({
        'texto': df['texto'].tolist(),
        'label': df['final_label'].map(LABEL2ID).tolist(),
    })
    return ds.map(tokenize, batched=True, remove_columns=['texto'])

print('Tokenizando datasets...')
ds_train = make_dataset(df_train)
ds_val   = make_dataset(df_val)

print(f'Train: {len(ds_train):,} ejemplos')
print(f'Val  : {len(ds_val):,} ejemplos')
print(f'Columnas: {ds_train.column_names}')


Tokenizando datasets...


Map:   0%|          | 0/1902 [00:00<?, ? examples/s]

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

Train: 1,902 ejemplos
Val  : 238 ejemplos
Columnas: ['label', 'input_ids', 'attention_mask']


## 8. Funcion de metricas

Las metricas que se calculan al final de cada epoca sobre el conjunto de validacion.

La metrica principal es F1-macro porque:
- Trata las tres clases (IZQUIERDA, CENTRO, DERECHA) por igual
- No se ve distorsionada por el desbalanceo de clases (CENTRO=40%, DERECHA=24%)
- Es el estandar en papers de clasificacion ideologica y de texto en general

El checkpoint que se guarda como mejor es el que maximiza F1-macro en validacion,no el del ultimo epoch.


In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc      = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro',    labels=[0,1,2], zero_division=0)
    f1_w     = f1_score(labels, preds, average='weighted', labels=[0,1,2], zero_division=0)
    kappa    = cohen_kappa_score(labels, preds)

    probs     = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
    confianza = probs.max(axis=1).mean()

    prec, rec, f1_per, _ = precision_recall_fscore_support(
        labels, preds, labels=[0,1,2], zero_division=0
    )

    return {
        'accuracy':    round(acc, 4),
        'f1_macro':    round(f1_macro, 4),
        'f1_weighted': round(f1_w, 4),
        'kappa':       round(kappa, 4),
        'confianza':   round(float(confianza), 4),
        'f1_izq':      round(f1_per[0], 4),
        'f1_cen':      round(f1_per[1], 4),
        'f1_der':      round(f1_per[2], 4),
        'prec_izq':    round(prec[0], 4),
        'prec_cen':    round(prec[1], 4),
        'prec_der':    round(prec[2], 4),
        'rec_izq':     round(rec[0], 4),
        'rec_cen':     round(rec[1], 4),
        'rec_der':     round(rec[2], 4),
    }

print('Funcion de metricas definida')


Funcion de metricas definida


## 9. TrainingArguments

Configuracion del entrenamiento.


In [15]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Epocas y batch
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,

    # Optimizacion
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=True,

    # Evaluacion y guardado
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,

    # Logging
    logging_steps=20,
    report_to='none',
    seed=42,
)

batch_efectivo = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
pasos_por_epoca = len(ds_train) // batch_efectivo
print(f'Batch efectivo  : {batch_efectivo}')
print(f'Pasos por epoca : {pasos_por_epoca}')
print(f'Total pasos     : {pasos_por_epoca * training_args.num_train_epochs}')


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Batch efectivo  : 32
Pasos por epoca : 59
Total pasos     : 590


## 10. Trainer y entrenamiento

Lanzamos el entrenamiento.

DataCollatorWithPadding realiza padding dinamico: padea cada batch solo hasta
el ejemplo mas largo de ese batch, no hasta max_length=1024. Esto ahorra VRAM
y acelera el entrenamiento cuando los articulos son de longitud variable.


In [16]:
!nvidia-smi

Tue May 19 17:38:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   36C    P0             88W /  600W |   96807MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [18]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9
print(f"VRAM libre tras limpieza: {free:.1f} GB")

VRAM libre tras limpieza: 86.4 GB


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

# Detectar checkpoint previo para reanudar si la sesion se corto
checkpoints = sorted(Path(OUTPUT_DIR).glob('checkpoint-*'))
last_checkpoint = str(checkpoints[-1]) if checkpoints else None

if last_checkpoint:
    print(f'Checkpoint encontrado: {last_checkpoint}')
    print('Reanudando entrenamiento desde ese punto...')
else:
    print('Iniciando entrenamiento desde cero...')

trainer.train(resume_from_checkpoint=last_checkpoint)

print('Entrenamiento completado')


Iniciando entrenamiento desde cero...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Kappa,Confianza,F1 Izq,F1 Cen,F1 Der,Prec Izq,Prec Cen,Prec Der,Rec Izq,Rec Cen,Rec Der
1,5.646607,0.621200,0.747900,0.740800,0.744900,0.604200,0.765700,0.738900,0.767900,0.715800,0.794500,0.671900,0.918900,0.690500,0.895800,0.586200
2,3.693013,0.520192,0.781500,0.776400,0.783300,0.670200,0.777500,0.764300,0.828000,0.736800,0.821900,0.855600,0.653300,0.714300,0.802100,0.844800
3,1.645892,0.483818,0.823500,0.826100,0.823300,0.731100,0.891400,0.804700,0.823500,0.850000,0.800000,0.846200,0.822600,0.809500,0.802100,0.879300
4,0.991998,1.018934,0.764700,0.756500,0.761200,0.639300,0.942300,0.722200,0.809500,0.737700,0.866700,0.745600,0.703100,0.619000,0.885400,0.775900
5,0.529403,1.078442,0.806700,0.801200,0.806100,0.701900,0.954400,0.822100,0.817700,0.763600,0.848100,0.775700,0.807700,0.797600,0.864600,0.724100
6,0.113856,1.737079,0.768900,0.762400,0.767600,0.647300,0.978600,0.743200,0.807900,0.736000,0.859400,0.766400,0.686600,0.654800,0.854200,0.793100
7,0.000704,1.297597,0.852900,0.848400,0.853500,0.776100,0.978900,0.869000,0.866300,0.809900,0.869000,0.890100,0.777800,0.869000,0.843800,0.844800
8,0.000208,1.430171,0.840300,0.837300,0.840000,0.755300,0.982100,0.855500,0.842100,0.814200,0.831500,0.851100,0.836400,0.881000,0.833300,0.793100
9,0.000020,1.419325,0.836100,0.832800,0.836000,0.748800,0.979600,0.852100,0.839400,0.807000,0.847100,0.835100,0.821400,0.857100,0.843800,0.793100
10,0.000004,1.412795,0.840300,0.836500,0.840100,0.755300,0.979100,0.858800,0.843800,0.807000,0.848800,0.843800,0.821400,0.869000,0.843800,0.793100


Entrenamiento completado


## 11. Métricas finales en validación

Evaluamos el mejor checkpoint (el que maximizó F1-macro en validación a lo largo del entrenamiento) sobre el conjunto completo de validación (238 artículos).

`load_best_model_at_end=True` garantiza que el modelo evaluado es el checkpoint óptimo, no el del último epoch.

In [ ]:
print('Evaluando en validacion...')
val_output = trainer.predict(ds_val)

logits = val_output.predictions
labels = val_output.label_ids
probs  = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
preds  = np.argmax(probs, axis=-1)

acc      = accuracy_score(labels, preds)
f1_macro = f1_score(labels, preds, average='macro',    labels=[0,1,2], zero_division=0)
f1_w     = f1_score(labels, preds, average='weighted', labels=[0,1,2], zero_division=0)
kappa    = cohen_kappa_score(labels, preds)
conf     = probs.max(axis=1).mean()

prec, rec, f1_per, _ = precision_recall_fscore_support(
    labels, preds, labels=[0,1,2], zero_division=0
)
cm = confusion_matrix(labels, preds, labels=[0,1,2])

sep = '=' * 50
print(f'\n{sep}')
print('METRICAS VALIDACION — Qwen3-4B LoRA')
print(sep)
print(f'  Accuracy    : {acc:.4f}')
print(f'  F1-macro    : {f1_macro:.4f}')
print(f'  F1-weighted : {f1_w:.4f}')
print(f'  Kappa       : {kappa:.4f}')
print(f'  Confianza   : {conf:.4f}')
for i, cls in enumerate(CLASES):
    print(f'  {cls}: P={prec[i]:.3f}  R={rec[i]:.3f}  F1={f1_per[i]:.3f}')
print(f'\n  Matriz de confusion (filas=real, cols=pred):')
cm_df = pd.DataFrame(cm, index=CLASES, columns=CLASES)
print(cm_df.to_string())


### Análisis de resultados en validación

| Métrica | Valor |
|---------|-------|
| Accuracy | 0.8529 |
| F1-macro | **0.8484** |
| F1-weighted | 0.8535 |
| Kappa de Cohen (κ) | 0.7761 |
| Confianza media | 0.9789 |

F1 por clase:

| Clase | F1 |
|-------|----|
| IZQUIERDA | 0.869 |
| CENTRO | 0.866 |
| DERECHA | 0.810 |

El kappa κ=0.7761 indica **acuerdo sustancial** entre las predicciones y el ground truth (por encima del umbral de 0.75). IZQUIERDA obtiene el F1 más alto de las tres clases en esta variante (0.869), resultado inusual dado que en el resto de modelos IZQUIERDA suele ser la clase más difícil. DERECHA queda con el F1 más bajo (0.810), coherente con su menor representación en el conjunto de entrenamiento (464 ejemplos frente a 766 de CENTRO).

La confianza media (0.9789) es la más alta de toda la batería. Sin embargo, esto no es necesariamente positivo: combinada con el overfitting elevado, resulta en que el 91.4 % de los errores del modelo se producen con confianza superior a 0.85 (errores de alta confianza). Un modelo bien calibrado debería mostrar menor confianza en los casos donde se equivoca.

## 12. Guardar predicciones, metricas y modelo final

Guardamos en Drive:
1. Predicciones de validacion con probabilidades por clase: necesarias para el reranking
   de la Seccion 6 y para la tabla comparativa final.
2. Metricas en CSV: una fila por modelo, mismo formato que zero-shot y few-shot.
3. El mejor modelo: para inferencia posterior sobre los 238K articulos sin etiquetar.

In [ ]:
# Predicciones de validacion con probabilidades
resultados_val = pd.DataFrame({
    'id':        df_val['id'].tolist(),
    'true':      [ID2LABEL[l] for l in labels],
    'pred':      [ID2LABEL[p] for p in preds],
    'p_izq':     probs[:, 0],
    'p_cen':     probs[:, 1],
    'p_der':     probs[:, 2],
    'confianza': probs.max(axis=1),
})
pred_path = OUTPUT_DIR + '/val_predictions.csv'
resultados_val.to_csv(pred_path, index=False)
print(f'Predicciones guardadas: {pred_path}')

# Metricas en CSV
metricas = {
    'modelo':      'Qwen3-4B_LoRA_mlp32',
    'accuracy':    round(acc, 4),
    'f1_macro':    round(f1_macro, 4),
    'f1_weighted': round(f1_w, 4),
    'kappa':       round(kappa, 4),
    'confianza':   round(float(conf), 4),
    'prec_izq':    round(prec[0], 4), 'rec_izq': round(rec[0], 4), 'f1_izq': round(f1_per[0], 4),
    'prec_cen':    round(prec[1], 4), 'rec_cen': round(rec[1], 4), 'f1_cen': round(f1_per[1], 4),
    'prec_der':    round(prec[2], 4), 'rec_der': round(rec[2], 4), 'f1_der': round(f1_per[2], 4),
}
metrics_path = OUTPUT_DIR + '/val_metrics.csv'
pd.DataFrame([metricas]).to_csv(metrics_path, index=False)
print(f'Metricas guardadas   : {metrics_path}')

# Modelo final
best_model_path = OUTPUT_DIR + '/best_model'
trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)
print(f'Modelo guardado      : {best_model_path}')

sep = '=' * 50
print(f'\n{sep}')
print('RESUMEN FINAL')
print(sep)
print(f'  F1-macro val : {f1_macro:.4f}')
print(f'  Kappa val    : {kappa:.4f}')
print(f'  Modelo       : {best_model_path}')


---
## Análisis del entrenamiento

Las secciones siguientes examinan la dinámica del aprendizaje mediante curvas de loss y F1, la interpretabilidad del modelo (SHAP, LIME) y la distribución de errores en validación.

## 13. Curvas de aprendizaje

Evolución del F1-macro, la validation loss y la confianza a lo largo de las 10 épocas de entrenamiento. El objetivo es:
- Detectar si hay sobreajuste: la val_loss sube mientras la train_loss sigue bajando
- Identificar el epoch óptimo (mayor F1-macro en validación)
- Comparar la velocidad de convergencia de las tres clases ideológicas

In [ ]:
log_history = trainer.state.log_history
val_logs = [x for x in log_history if 'eval_f1_macro' in x]

if val_logs:
    epocas   = [x['epoch'] for x in val_logs]
    f1_vals  = [x['eval_f1_macro'] for x in val_logs]
    acc_vals = [x['eval_accuracy'] for x in val_logs]
    mejor_epoca = epocas[f1_vals.index(max(f1_vals))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epocas, f1_vals, 'o-', color='#2166ac', linewidth=2, markersize=6)
    axes[0].axvline(mejor_epoca, color='#d6604d', linestyle='--', alpha=0.8,
                    label=f'Mejor epoca {mejor_epoca:.0f} (F1={max(f1_vals):.4f})')
    axes[0].set_xlabel('Epoca')
    axes[0].set_ylabel('F1-macro')
    axes[0].set_title('F1-macro en validacion')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epocas, acc_vals, 'o-', color='#4dac26', linewidth=2, markersize=6)
    axes[1].set_xlabel('Epoca')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy en validacion')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Qwen3-4B_LoRA_mlp32 — Curvas de aprendizaje', fontsize=13)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR + '/learning_curves.png', dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Grafica guardada: {OUTPUT_DIR}/learning_curves.png')
else:
    print('No hay logs de validacion disponibles')


In [ ]:
log_history = trainer.state.log_history
val_logs    = [x for x in log_history if 'eval_f1_macro' in x]
train_logs  = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]

epocas     = [x['epoch'] for x in val_logs]
val_loss   = [x['eval_loss'] for x in val_logs]
f1_macro   = [x['eval_f1_macro'] for x in val_logs]
f1_izq     = [x['eval_f1_izq'] for x in val_logs]
f1_cen     = [x['eval_f1_cen'] for x in val_logs]
f1_der     = [x['eval_f1_der'] for x in val_logs]
kappa_vals = [x['eval_kappa'] for x in val_logs]
confianza  = [x['eval_confianza'] for x in val_logs]

pasos_train      = [x['step'] for x in train_logs]
train_loss_pasos = [x['loss'] for x in train_logs]

n_epocas = len(epocas)
pasos_por_epoca = len(train_loss_pasos) // n_epocas
train_loss = [
    float(np.mean(train_loss_pasos[i*pasos_por_epoca:(i+1)*pasos_por_epoca]))
    for i in range(n_epocas)
]

mejor_epoca = epocas[f1_macro.index(max(f1_macro))]
mejor_f1    = max(f1_macro)

print("Train loss por época:", [f'{x:.4f}' for x in train_loss])
print("Val   loss por época:", [f'{x:.4f}' for x in val_loss])
print("Gap   (val-train)   :", [f'{v-t:.4f}' for v,t in zip(val_loss, train_loss)])

In [ ]:
print('\n1. DIAGNOSTICO DE OVERFITTING')
print('-' * 40)
for i, e in enumerate(epocas):
    gap = val_loss[i] - train_loss[i] if i < len(train_loss) else 0
    estado = ''
    if i == 0:
        estado = 'aprendiendo'
    elif f1_macro[i] > f1_macro[i-1]:
        estado = 'mejorando'
    elif abs(f1_macro[i] - f1_macro[i-1]) < 0.005:
        estado = 'estabilizado'
    else:
        estado = 'OVERFITTING'
    marca = '<-- MEJOR' if e == mejor_epoca else ''
    print(f'  Epoca {e:.0f}: Val Loss {val_loss[i]:.4f}  F1 {f1_macro[i]:.4f}  [{estado}] {marca}')

In [ ]:
print('    Diagnóstico overfitting basado en val_loss y F1\n')

PATIENCE  = 3
MIN_DELTA = 0.005
GAP_ALERT = 0.3

mejor_vl              = float('inf')
mejor_vl_epoca        = 1
epocas_sin_mejora     = 0

for i, e in enumerate(epocas):

    gap = val_loss[i] - train_loss[i]

    # Actualizar mejor val_loss
    if val_loss[i] < mejor_vl - MIN_DELTA:
        mejor_vl          = val_loss[i]
        mejor_vl_epoca    = e
        epocas_sin_mejora = 0
    else:
        epocas_sin_mejora += 1

    # Cambio de F1
    if i == 0:
        cambio_f1 = None
        f1_str    = ''
    else:
        cambio_f1 = f1_macro[i] - f1_macro[i-1]
        signo     = '+' if cambio_f1 >= 0 else ''
        f1_str    = f'F1 {signo}{cambio_f1:.4f}'

    # usa GAP como señal principal
    if i == 0:
        estado = 'aprendiendo'
    elif epocas_sin_mejora >= PATIENCE and gap > GAP_ALERT:
        estado = 'OVERFITTING CONFIRMADO'       # val_loss sube + gap grande
    elif epocas_sin_mejora >= PATIENCE:
        estado = 'val_loss estancada'            # sin mejorar pero gap ok
    elif gap > GAP_ALERT:
        estado = 'señal overfitting (gap alto)'  # gap grande pero aún no confirmado
    elif cambio_f1 and cambio_f1 > 0:
        estado = 'mejorando'
    else:
        estado = 'estabilizado'

    marca    = '<-- MEJOR F1'    if e == mejor_epoca   else ''
    marca_vl = '<-- MEJOR VAL'   if e == mejor_vl_epoca else ''

    print(f'  Época {e:2.0f} | '
          f'Train {train_loss[i]:.4f} | '
          f'Val {val_loss[i]:.4f} | '
          f'Gap {gap:+.4f} | '
          f'F1 {f1_macro[i]:.4f} ({f1_str:12s}) | '
          f'{estado:35s} {marca} {marca_vl}')

In [ ]:
print('\n2. EVOLUCION F1 POR CLASE')
print('-' * 40)
print(f'  {"Epoca":<8} {"F1-IZQ":<10} {"F1-CEN":<10} {"F1-DER":<10} {"F1-MACRO":<10}')
for i, e in enumerate(epocas):
    marca = ' <-- MEJOR' if e == mejor_epoca else ''
    print(f'  {e:<8.0f} {f1_izq[i]:<10.4f} {f1_cen[i]:<10.4f} {f1_der[i]:<10.4f} {f1_macro[i]:<10.4f}{marca}')


In [ ]:
print('\n3. EVOLUCION DE CONFIANZA DEL MODELO')
print('-' * 40)
for i, e in enumerate(epocas):
    barra = '#' * int(confianza[i] * 20)
    print(f'  Epoca {e:.0f}: {confianza[i]:.4f} |{barra}|')

In [ ]:
print('\n4. CONCLUSION')
print('-' * 40)
epoca_pico = epocas[f1_macro.index(max(f1_macro))]
f1_final   = f1_macro[-1]
degradacion = max(f1_macro) - f1_final

if degradacion > 0.02:
    print(f'  Overfitting detectado a partir de la epoca {epoca_pico:.0f}.')
    print(f'  El F1-macro cae {degradacion:.4f} puntos desde el pico hasta la epoca final.')
    print(f'  El checkpoint optimo es la epoca {epoca_pico:.0f} (F1-macro={mejor_f1:.4f}).')
    print(f'  load_best_model_at_end=True ha seleccionado este checkpoint automaticamente.')
else:
    print(f'  No se detecta overfitting significativo.')
    print(f'  El modelo se estabiliza en torno a F1-macro={mejor_f1:.4f}.')


In [ ]:
# Graficas
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1 — Training Loss por paso
axes[0,0].plot(pasos_train, train_loss_pasos, color='#d6604d', linewidth=1, alpha=0.8)
axes[0,0].set_xlabel('Paso')
axes[0,0].set_ylabel('Training Loss')
axes[0,0].set_title('Training Loss por paso')
axes[0,0].grid(True, alpha=0.3)

#2 — Val Loss vs Training Loss por epoca
axes[0,1].plot(epocas, val_loss, 'o-', color='#2166ac', linewidth=2, markersize=6, label='Validation Loss')
axes[0,1].axvline(mejor_epoca, color='#4dac26', linestyle='--', alpha=0.8, label=f'Mejor epoca {mejor_epoca:.0f}')
axes[0,1].set_xlabel('Epoca')
axes[0,1].set_ylabel('Loss')
axes[0,1].set_title('Validation Loss por epoca')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3 — F1 por clase
axes[1,0].plot(epocas, f1_izq, 'o-', color='#d6604d',  linewidth=2, markersize=5, label='IZQUIERDA')
axes[1,0].plot(epocas, f1_cen, 's-', color='#4dac26',  linewidth=2, markersize=5, label='CENTRO')
axes[1,0].plot(epocas, f1_der, '^-', color='#2166ac',  linewidth=2, markersize=5, label='DERECHA')
axes[1,0].plot(epocas, f1_macro, 'D-', color='black',  linewidth=2, markersize=5, label='F1-macro', linestyle='--')
axes[1,0].axvline(mejor_epoca, color='gray', linestyle=':', alpha=0.8)
axes[1,0].set_xlabel('Epoca')
axes[1,0].set_ylabel('F1')
axes[1,0].set_title('F1 por clase y F1-macro en validacion')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 4 — Kappa y Confianza
ax2 = axes[1,1].twinx()
axes[1,1].plot(epocas, kappa_vals, 'o-', color='#762a83', linewidth=2, markersize=6, label='Kappa')
ax2.plot(epocas, confianza, 's--', color='#e08214', linewidth=2, markersize=6, label='Confianza')
axes[1,1].axvline(mejor_epoca, color='gray', linestyle=':', alpha=0.8)
axes[1,1].set_xlabel('Epoca')
axes[1,1].set_ylabel('Kappa', color='#762a83')
ax2.set_ylabel('Confianza', color='#e08214')
axes[1,1].set_title('Kappa y Confianza por epoca')
axes[1,1].grid(True, alpha=0.3)

lines1, labels1 = axes[1,1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1,1].legend(lines1 + lines2, labels1 + labels2)

plt.suptitle('Qwen3-4B_LoRA_mlp32 — Analisis completo del aprendizaje', fontsize=14)
plt.tight_layout()
fig.savefig(OUTPUT_DIR + '/analisis_aprendizaje.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'\nGrafica guardada: {OUTPUT_DIR}/analisis_aprendizaje.png')


### Interpretación de las curvas de aprendizaje

**Panel superior izquierdo — Training Loss por paso:**
La curva de training loss desciende de forma continua a lo largo de los 590 pasos (10 épocas × 59 pasos/época con batch efectivo de 32). Una caída suave indica que el learning rate (2e-4) y el scheduler de warmup son adecuados.

**Panel superior derecho — Validation Loss por época:**
La val_loss cae inicialmente, alcanza un mínimo (0.484) en la época de mejor F1, y luego sube hasta 1.413 al final del entrenamiento. Esta subida de 0.929 puntos clasifica esta variante como **overfitting BASTANTE** según el diagnóstico cruzado del notebook comparativa. `load_best_model_at_end=True` recupera automáticamente el checkpoint óptimo.

**Panel inferior izquierdo — F1 por clase:**
Los F1 finales en validación son IZQUIERDA 0.869, CENTRO 0.866 y DERECHA 0.810. IZQUIERDA alcanza aquí su mejor resultado relativo de toda la batería. DERECHA queda por debajo de las otras dos clases, coherente con su menor representación en el dataset de entrenamiento.

**Panel inferior derecho — Kappa y Confianza:**
El kappa de Cohen (0.7761) indica acuerdo sustancial con el ground truth. La confianza media final (0.9789) es la más alta de toda la batería: el modelo es muy seguro de sus predicciones, incluso cuando se equivoca, lo que es una consecuencia directa del overfitting elevado.

## 14. Interpretabilidad con SHAP

SHAP (SHapley Additive exPlanations) cuantifica la contribución de cada token del texto a la predicción del modelo para cada clase ideológica, calculando el valor marginal de cada token respecto a todas las posibles permutaciones.

Se usa PartitionExplainer porque trabaja directamente sobre texto con particiones jerárquicas de palabras, sin necesidad de construir un fondo de referencia vectorial, lo que lo hace más eficiente que KernelExplainer para modelos de lenguaje de este tamaño.

Salida: barplot con los 20 tokens más influyentes por clase y visualización HTML de un artículo coloreado por valor SHAP.

In [ ]:
N_SHAP_EXPLAIN  = 15   # artículos de val a explicar (más = más lento)
MAX_LEN_SHAP    = 256  # longitud máxima para SHAP (equilibrio velocidad/precisión)
TOP_K_TOKENS    = 20   # tokens más influyentes a mostrar por clase

model.eval()

def predict_text(texts):
    """Devuelve probabilidades (n, 3) para una lista de strings."""
    all_probs = []
    for i in range(0, len(texts), 8):
        batch = list(texts[i:i+8])
        enc = tokenizer(
            batch,
            truncation=True,
            max_length=MAX_LEN_SHAP,
            padding=True,
            return_tensors='pt'
        )
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)

# Textos a explicar: primeros N_SHAP_EXPLAIN de val
textos_shap = df_val['texto'].iloc[:N_SHAP_EXPLAIN].tolist()
labels_shap = df_val['final_label'].iloc[:N_SHAP_EXPLAIN].tolist()


masker = shap.maskers.Text(tokenizer=r"\W+")
explainer = shap.Explainer(predict_text, masker, output_names=CLASES)

print(f"Calculando SHAP values para {N_SHAP_EXPLAIN} artículos...")
print("(Esto puede tardar 5-10 minutos en Colab T4)")
shap_values = explainer(textos_shap, batch_size=4)
print("SHAP values calculados.")

# Guardar SHAP values
shap_path = OUTPUT_DIR + '/shap_values.npz'
np.savez(shap_path,
         values=shap_values.values,
         data=np.array(shap_values.data, dtype=object),
         labels=np.array(labels_shap))
print(f"SHAP values guardados: {shap_path}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8))
colores_clase = {'IZQUIERDA': '#d6604d', 'CENTRO': '#4dac26', 'DERECHA': '#2166ac'}

for cls_idx, cls_name in enumerate(CLASES):
    token_shap = {}
    for i in range(len(textos_shap)):
        tokens_i = shap_values[i].data
        vals_i   = shap_values[i].values[:, cls_idx]
        for tok, val in zip(tokens_i, vals_i):
            tok_clean = tok.strip().lower()
            if len(tok_clean) < 2:
                continue
            if tok_clean not in token_shap:
                token_shap[tok_clean] = []
            token_shap[tok_clean].append(val)

    # Media de SHAP por token
    token_mean = {t: np.mean(v) for t, v in token_shap.items() if len(v) >= 1}
    df_tok = pd.DataFrame(list(token_mean.items()), columns=['token', 'shap'])
    df_tok = df_tok.reindex(df_tok['shap'].abs().nlargest(TOP_K_TOKENS).index)
    df_tok = df_tok.sort_values('shap')

    color_bars = [colores_clase[cls_name] if v > 0 else '#aaaaaa' for v in df_tok['shap']]
    axes[cls_idx].barh(df_tok['token'], df_tok['shap'], color=color_bars, edgecolor='white')
    axes[cls_idx].axvline(0, color='black', linewidth=0.8)
    axes[cls_idx].set_title(f'Top tokens — {cls_name}', fontsize=13, fontweight='bold')
    axes[cls_idx].set_xlabel('SHAP value medio')
    axes[cls_idx].grid(True, axis='x', alpha=0.3)

plt.suptitle('Qwen3-4B_LoRA_mlp32 — Tokens más influyentes por clase (SHAP)', fontsize=14)
plt.tight_layout()
fig.savefig(OUTPUT_DIR + '/shap_top_tokens.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR}/shap_top_tokens.png")

### Interpretación del análisis SHAP

Los barplots muestran los 20 tokens con mayor valor SHAP absoluto para cada clase, calculados como media sobre 15 artículos de validación.

Un SHAP positivo (barras de color) indica que ese token empuja la predicción hacia esa clase; un SHAP negativo (barras grises) indica que la reduce. Los valores se normalizan por artículo para hacer las comparaciones entre clases más legibles.

Claves de interpretación:
- Un token que aparece con SHAP alto en IZQUIERDA y negativo en DERECHA es un marcador ideológico discriminativo robusto
- Si un mismo token tiene SHAP positivo en CENTRO y negativo en IZQUIERDA o DERECHA, actúa como moderador semántico
- Con r=32 en las MLP, esta variante dispone de mayor capacidad de representación, lo que puede producir valores SHAP distribuidos sobre un vocabulario más amplio y menos concentrado en los términos ideológicos más obvios

Limitación: los valores SHAP se calculan sobre 15 artículos de validación para mantener un tiempo de ejecución razonable. La generalización al corpus completo debe hacerse con cautela.

In [ ]:
IDX_EJEMPLO = 0
CLS_VER     = 2   # 0=IZQUIERDA, 1=CENTRO, 2=DERECHA

sv   = shap_values[IDX_EJEMPLO].values[:, CLS_VER]
toks = shap_values[IDX_EJEMPLO].data

# Normalizar a [-1, 1] para el colormap
vmax = max(abs(sv.max()), abs(sv.min())) + 1e-9
sv_norm = sv / vmax

cmap_pos = plt.cm.Reds
cmap_neg = plt.cm.Blues

html_parts = [f"<div style='font-family:monospace;font-size:13px;line-height:2;padding:12px;background:#fafafa;border:1px solid #ddd;border-radius:6px;'>"]
html_parts.append(f"<b>Artículo #{IDX_EJEMPLO} | Label real: {labels_shap[IDX_EJEMPLO]} | Clase visualizada: {CLASES[CLS_VER]}</b><br><br>")

for tok, val in zip(toks, sv_norm):
    if val > 0:
        rgba = cmap_pos(0.2 + 0.8 * val)
    else:
        rgba = cmap_neg(0.2 + 0.8 * abs(val))
    hex_color = mcolors.to_hex(rgba)
    text_color = '#000' if val < 0.6 else '#fff'
    html_parts.append(
        f"<span style='background:{hex_color};color:{text_color};"
        f"padding:2px 3px;margin:1px;border-radius:3px;' "
        f"title='SHAP={val*vmax:.4f}'>{tok}</span>"
    )

html_parts.append("</div>")
html_output = "".join(html_parts)
display(HTML(html_output))
print(f"\nRojo = empuja hacia {CLASES[CLS_VER]} | Azul = empuja en contra")

## 15. Análisis de errores cualitativos

La matriz de confusión indica cuántos errores hay; esta sección responde cuáles y por qué.

Tres perspectivas:
1. **Errores por tipo de confusión**: qué pares de clases se confunden entre sí
2. **Casos con mayor pérdida cross-entropy**: los artículos más difíciles para el modelo
3. **Errores con alta confianza (> 0.85)**: el modelo equivocado y muy seguro de estarlo, los más problemáticos para el reranking posterior

En clasificación ideológica los errores IZQUIERDA-DERECHA son los más graves semánticamente. Los errores IZQUIERDA-CENTRO o CENTRO-DERECHA son más esperables dado que CENTRO es una categoría inherentemente difusa.

In [ ]:
rv = pd.read_csv(OUTPUT_DIR + '/val_predictions.csv')
rv['correcto']  = rv['true'] == rv['pred']
rv['longitud']  = df_val['texto'].str.len().values
rv['n_tokens']  = [len(tokenizer.encode(t, truncation=True, max_length=1024)) for t in df_val['texto'].tolist()]
rv['truncado']  = rv['n_tokens'] >= 1023

probs_mat = rv[['p_izq','p_cen','p_der']].values
label_ids = rv['true'].map(LABEL2ID).values
log_p     = np.log(probs_mat + 1e-9)
rv['loss_ce'] = -log_p[np.arange(len(label_ids)), label_ids]

sep = '=' * 60
print(sep)
print('ANÁLISIS DE ERRORES CUALITATIVOS — Qwen3-4B LoRA')
print(sep)

#Errores por tipo de confusión
errores = rv[~rv['correcto']].copy()
print(f'\n1. ERRORES TOTALES: {len(errores)} / {len(rv)} artículos')
print('-' * 50)
confusion_tipos = errores.groupby(['true','pred']).size().reset_index(name='n')
confusion_tipos = confusion_tipos.sort_values('n', ascending=False)
for _, row in confusion_tipos.iterrows():
    alerta = ' ALERTA IDEOLÓGICA' if set([row['true'],row['pred']]) == {'IZQUIERDA','DERECHA'} else ''
    print(f"  {row['true']:12} → {row['pred']:12}  ({row['n']} casos){alerta}")

# Casos con mayor pérdida (más difíciles para el modelo)
N_HARD = 5
print(f'\n2. TOP-{N_HARD} ARTÍCULOS CON MAYOR PÉRDIDA (más difíciles)')
print('-' * 50)
hard_cases = rv.nlargest(N_HARD, 'loss_ce')
for _, row in hard_cases.iterrows():
    print(f"  ID: {row['id']}  | Real: {row['true']:10} Pred: {row['pred']:10} | Loss: {row['loss_ce']:.3f} | Conf: {row['confianza']:.3f}")
    texto_preview = df_val[df_val['id']==row['id']]['texto'].values
    if len(texto_preview) > 0:
        print(f"  Texto (primeros 200 chars): {texto_preview[0][:200]}...")
    print()

# Errores con alta confianza (peligrosos para el reranking)
UMBRAL_CONFIANZA = 0.85
print(f'\n3. ERRORES CON CONFIANZA > {UMBRAL_CONFIANZA} (peligrosos para reranking)')
print('-' * 50)
high_conf_errors = errores[errores['confianza'] > UMBRAL_CONFIANZA].sort_values('confianza', ascending=False)
print(f'  Errores con alta confianza: {len(high_conf_errors)} / {len(errores)} ({100*len(high_conf_errors)/max(len(errores),1):.1f}%)')
if len(high_conf_errors) > 0:
    for _, row in high_conf_errors.head(5).iterrows():
        print(f"  ID: {row['id']} | Real: {row['true']:10} Pred: {row['pred']:10} | Conf: {row['confianza']:.3f}")
else:
    print('  No hay errores con confianza > {:.2f}. El modelo es conservador en sus errores.'.format(UMBRAL_CONFIANZA))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfica 1: Confianza en aciertos vs errores
conf_ok  = rv[rv['correcto']]['confianza']
conf_err = rv[~rv['correcto']]['confianza']

axes[0].hist(conf_ok,  bins=20, alpha=0.7, color='#4dac26', label=f'Aciertos (n={len(conf_ok)})',  density=True)
axes[0].hist(conf_err, bins=20, alpha=0.7, color='#d6604d', label=f'Errores (n={len(conf_err)})',   density=True)
axes[0].axvline(conf_ok.mean(),  color='#4dac26', linestyle='--', linewidth=2, label=f'Media aciertos: {conf_ok.mean():.3f}')
axes[0].axvline(conf_err.mean(), color='#d6604d', linestyle='--', linewidth=2, label=f'Media errores:  {conf_err.mean():.3f}')
axes[0].set_xlabel('Confianza')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Confianza: aciertos vs errores')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Gráfica 2: Longitud del texto vs correcto/error
axes[1].scatter(rv[rv['correcto']]['longitud'],  rv[rv['correcto']]['loss_ce'],
                c='#4dac26', alpha=0.4, s=20, label='Acierto')
axes[1].scatter(rv[~rv['correcto']]['longitud'], rv[~rv['correcto']]['loss_ce'],
                c='#d6604d', alpha=0.8, s=40, marker='X', label='Error')
axes[1].set_xlabel('Longitud texto (caracteres)')
axes[1].set_ylabel('Pérdida cross-entropy')
axes[1].set_title('Longitud vs pérdida por ejemplo')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Gráfica 3: Pérdida por clase real
clases_col = {'IZQUIERDA': '#d6604d', 'CENTRO': '#4dac26', 'DERECHA': '#2166ac'}
for cls in CLASES:
    datos_cls = rv[rv['true']==cls]['loss_ce']
    axes[2].hist(datos_cls, bins=15, alpha=0.6, color=clases_col[cls],
                 label=f'{cls} (μ={datos_cls.mean():.3f})', density=True)
axes[2].set_xlabel('Pérdida cross-entropy')
axes[2].set_ylabel('Densidad')
axes[2].set_title('Distribución de pérdida por clase')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Qwen3-4B_LoRA_mlp32 — Análisis de errores cualitativos', fontsize=13)
plt.tight_layout()
fig.savefig(OUTPUT_DIR + '/error_analysis.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR}/error_analysis.png")

### Interpretación del análisis de errores

**Gráfica izquierda — Confianza en aciertos vs errores:**
Esta variante presenta la confianza media más alta de toda la batería (0.9789). Si la distribución de los errores (rojo) se solapa con la de los aciertos (verde) en valores altos de confianza, el modelo no distingue cuándo va a equivocarse. El 91.4 % de los errores se producen con confianza superior a 0.85, lo que indica sobreconfianza como consecuencia del overfitting elevado.

**Gráfica central — Longitud vs pérdida:**
Cada punto es un artículo; los cruces rojos son errores. Si los errores se concentran en artículos muy largos (muchos tokens truncados), el límite de 1,024 tokens puede ser un factor contribuyente. Si se distribuyen uniformemente, el error es independiente de la longitud del artículo.

**Gráfica derecha — Pérdida por clase real:**
Una distribución más ancha en DERECHA es esperable porque es la clase con menos representación en el conjunto de entrenamiento (464 ejemplos frente a 766 de CENTRO y 672 de IZQUIERDA). Una cola derecha larga indica artículos frontera difíciles de clasificar.

Los errores IZQUIERDA-DERECHA son los más problemáticos semánticamente porque representan una confusión entre los extremos del espectro ideológico.

## 16. LIME: Explicaciones a nivel de palabras

LIME (Local Interpretable Model-agnostic Explanations) complementa SHAP trabajando a nivel de palabras completas en lugar de subwords del tokenizer, produciendo explicaciones más directamente legibles.

SHAP vs LIME:
- SHAP: riguroso matemáticamente (valores de Shapley exactos), granularidad de token, más costoso computacionalmente
- LIME: aproximación local por perturbaciones, granularidad de palabra, más rápido

Se explica un artículo por cada clase ideológica, seleccionando en cada caso el artículo clasificado correctamente con mayor confianza para capturar los patrones que el modelo ha aprendido de forma más robusta.

In [ ]:
def predict_lime(texts):
    """Devuelve probabilidades (n, 3) para lista de strings."""
    all_probs = []
    for i in range(0, len(texts), 8):
        batch = list(texts[i:i+8])
        enc = tokenizer(batch, truncation=True, max_length=256,
                        padding=True, return_tensors='pt')
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.vstack(all_probs)

explainer_lime = LimeTextExplainer(class_names=CLASES, random_state=42)

# Seleccionar 1 ejemplo por clase (aciertos con mayor confianza) ─
rv = pd.read_csv(OUTPUT_DIR + '/val_predictions.csv')
rv['correcto'] = rv['true'] == rv['pred']
rv['longitud'] = df_val['texto'].str.len().values

ejemplos_lime = {}
for cls in CLASES:
    candidatos = rv[(rv['true']==cls) & rv['correcto']].sort_values('confianza', ascending=False)
    if len(candidatos) > 0:
        idx_val = candidatos.index[0]
        ejemplos_lime[cls] = {
            'texto': df_val['texto'].iloc[idx_val],
            'id':    rv.loc[idx_val, 'id'],
            'conf':  rv.loc[idx_val, 'confianza'],
        }

# Generar explicaciones
lime_results = {}
print("Generando explicaciones LIME (puede tardar 2-3 min)...")
for cls, info in ejemplos_lime.items():
    texto_truncado = info['texto'][:2000]
    exp = explainer_lime.explain_instance(
        texto_truncado,
        predict_lime,
        num_features=15,
        num_samples=200,
        labels=[LABEL2ID[cls]]
    )
    lime_results[cls] = exp
    print(f"  {cls}: OK")

print("Explicaciones LIME generadas.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
colores_clase = {'IZQUIERDA': '#d6604d', 'CENTRO': '#4dac26', 'DERECHA': '#2166ac'}

for ax, (cls_name, exp) in zip(axes, lime_results.items()):
    cls_id  = LABEL2ID[cls_name]
    feats   = exp.as_list(label=cls_id)
    feats_sorted = sorted(feats, key=lambda x: x[1])
    palabras = [f[0] for f in feats_sorted]
    vals     = [f[1] for f in feats_sorted]
    colores  = [colores_clase[cls_name] if v > 0 else '#aaaaaa' for v in vals]

    ax.barh(palabras, vals, color=colores, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'LIME — {cls_name}\n(ID: {ejemplos_lime[cls_name]["id"]}, Conf: {ejemplos_lime[cls_name]["conf"]:.3f})',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Importancia LIME')
    ax.grid(True, axis='x', alpha=0.3)
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('Qwen3-4B_LoRA_mlp32 — Explicaciones LIME por clase', fontsize=13)
plt.tight_layout()
fig.savefig(OUTPUT_DIR + '/lime_explanations.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR}/lime_explanations.png")

### Interpretación de las explicaciones LIME

Cada barplot muestra las 15 palabras con mayor importancia LIME para la clase correspondiente en un artículo clasificado correctamente con alta confianza.

Las barras de color indican que la presencia de esa palabra aumenta la probabilidad de la clase asignada; las barras grises indican que la reduce. Los valores son locales al artículo concreto y no son comparables entre artículos distintos.

Para la clasificación ideológica de prensa española cabe esperar:
- **IZQUIERDA**: términos vinculados a servicios públicos, desigualdad, movimientos sociales, crítica al poder económico
- **CENTRO**: léxico neutral o institucional, menciones a acuerdos y pactos, terminología técnico-económica
- **DERECHA**: términos de orden, seguridad, identidad nacional, crítica al gobierno, economía de mercado

Con r=32 en las capas MLP esta variante tiene mayor capacidad para capturar vocabulario ideológico matizado, aunque a riesgo de memorizar términos específicos del conjunto de entrenamiento.

## 17. Análisis de longitud vs. rendimiento

El 45 % de los artículos de validación (107 de 238) supera el límite de 1,024 tokens. La longitud media sin truncar es de 1,093 tokens (mediana 986, percentil 95 de 2,056 tokens).

Esta sección responde si el truncado penaliza el rendimiento comparando artículos cortos (no truncados) frente a largos (truncados), y si existe una relación entre longitud y confianza del modelo.

In [ ]:
rv = pd.read_csv(OUTPUT_DIR + '/val_predictions.csv')
rv['correcto'] = rv['true'] == rv['pred']
rv['longitud_chars']  = df_val['texto'].str.len().values
rv['n_tokens']        = [len(tokenizer.encode(t, truncation=False)) for t in df_val['texto'].tolist()]
rv['n_tokens_trunc']  = rv['n_tokens'].clip(upper=1024)
rv['pct_truncado']    = ((rv['n_tokens'] - rv['n_tokens_trunc']) / rv['n_tokens'].clip(lower=1)).clip(lower=0)
rv['truncado']        = rv['n_tokens'] > 1023

label_ids = rv['true'].map(LABEL2ID).values
pred_ids  = rv['pred'].map(LABEL2ID).values

sep = '=' * 60
print(sep)
print('ANÁLISIS DE LONGITUD VS. RENDIMIENTO — Qwen3-4B LoRA')
print(sep)

print(f'\nArticulos truncados (> 1023 tokens): {rv["truncado"].sum()} / {len(rv)} ({100*rv["truncado"].mean():.1f}%)')
print(f'Longitud media (tokens, sin truncar): {rv["n_tokens"].mean():.0f}')
print(f'Longitud mediana: {rv["n_tokens"].median():.0f}  |  p95: {rv["n_tokens"].quantile(0.95):.0f}')

print('\nComparativa truncados vs no truncados:')
for truncado, grupo_name in [(False, 'No truncados'), (True, 'Truncados (>1023 tok)')]:
    sub = rv[rv['truncado']==truncado]
    if len(sub) == 0:
        continue
    acc  = sub['correcto'].mean()
    f1m  = f1_score(sub['true'].map(LABEL2ID), sub['pred'].map(LABEL2ID),
                    average='macro', labels=[0,1,2], zero_division=0)
    print(f'  {grupo_name:30} n={len(sub):3}  accuracy={acc:.4f}  F1-macro={f1m:.4f}')

# Análisis por cuartiles de longitud (tokens)
print('\nF1-macro por cuartil de longitud (tokens):')
rv['cuartil_tokens'] = pd.qcut(rv['n_tokens'], q=4,
                                labels=['Q1 (corto)', 'Q2', 'Q3', 'Q4 (largo)'])
for q in ['Q1 (corto)', 'Q2', 'Q3', 'Q4 (largo)']:
    sub = rv[rv['cuartil_tokens']==q]
    f1m = f1_score(sub['true'].map(LABEL2ID), sub['pred'].map(LABEL2ID),
                   average='macro', labels=[0,1,2], zero_division=0)
    rango = f"[{sub['n_tokens'].min():.0f}-{sub['n_tokens'].max():.0f} tok]"
    print(f'  {q:15} {rango:22} n={len(sub):3}  F1-macro={f1m:.4f}  acc={sub["correcto"].mean():.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Gráfica 1: Distribución de longitud en tokens
axes[0,0].hist(rv['n_tokens'], bins=30, color='#4dac26', alpha=0.7, edgecolor='white')
axes[0,0].axvline(1024, color='#d6604d', linestyle='--', linewidth=2, label='Límite truncado (1024)')
axes[0,0].axvline(rv['n_tokens'].mean(), color='gray', linestyle=':', linewidth=1.5, label=f"Media: {rv['n_tokens'].mean():.0f}")
axes[0,0].set_xlabel('Longitud (tokens)')
axes[0,0].set_ylabel('Número de artículos')
axes[0,0].set_title('Distribución de longitud en tokens')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Gráfica 2: Confianza vs n_tokens
colores_correcto = rv['correcto'].map({True: '#4dac26', False: '#d6604d'})
axes[0,1].scatter(rv['n_tokens'], rv['confianza'], c=colores_correcto, alpha=0.4, s=20)
axes[0,1].axvline(1024, color='#d6604d', linestyle='--', linewidth=1.5, label='Límite truncado')
axes[0,1].set_xlabel('Longitud (tokens, sin truncar)')
axes[0,1].set_ylabel('Confianza del modelo')
axes[0,1].set_title('Confianza vs longitud (verde=acierto, rojo=error)')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Gráfica 3: F1-macro por cuartil de longitud
cuartiles  = ['Q1 (corto)', 'Q2', 'Q3', 'Q4 (largo)']
f1_por_q   = []
acc_por_q  = []
for q in cuartiles:
    sub = rv[rv['cuartil_tokens']==q]
    f1_por_q.append(f1_score(sub['true'].map(LABEL2ID), sub['pred'].map(LABEL2ID),
                              average='macro', labels=[0,1,2], zero_division=0))
    acc_por_q.append(sub['correcto'].mean())

x = np.arange(len(cuartiles))
axes[1,0].bar(x - 0.2, f1_por_q,  0.35, label='F1-macro',  color='#2166ac', alpha=0.8)
axes[1,0].bar(x + 0.2, acc_por_q, 0.35, label='Accuracy', color='#762a83', alpha=0.8)
axes[1,0].axhline(rv['correcto'].mean(), color='gray', linestyle='--', linewidth=1, label='Acc global')
axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(cuartiles)
axes[1,0].set_ylabel('Métrica')
axes[1,0].set_title('F1-macro y Accuracy por cuartil de longitud')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3, axis='y')
axes[1,0].set_ylim(0.5, 1.0)

# Gráfica 4: % truncado vs confianza
axes[1,1].scatter(rv['pct_truncado'] * 100, rv['confianza'],
                   c=colores_correcto, alpha=0.4, s=20)
axes[1,1].set_xlabel('% de texto truncado')
axes[1,1].set_ylabel('Confianza del modelo')
axes[1,1].set_title('Porcentaje truncado vs confianza')
axes[1,1].grid(True, alpha=0.3)

# Añadir línea de tendencia
mask_valid = rv['pct_truncado'] > 0
if mask_valid.sum() > 5:
    z = np.polyfit(rv.loc[mask_valid,'pct_truncado']*100, rv.loc[mask_valid,'confianza'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, rv['pct_truncado'].max()*100, 50)
    axes[1,1].plot(x_line, p(x_line), 'k--', linewidth=1.5, alpha=0.6, label='Tendencia')
    axes[1,1].legend()

plt.suptitle('Qwen3-4B_LoRA_mlp32 — Análisis de longitud vs rendimiento', fontsize=13)
plt.tight_layout()
fig.savefig(OUTPUT_DIR + '/length_analysis.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR}/length_analysis.png")

print('\CONCLUSIÓN:')
delta_f1 = f1_por_q[-1] - f1_por_q[0]
if delta_f1 < -0.03:
    print(f'  Los artículos largos (Q4) tienen F1 {abs(delta_f1):.3f} puntos peor que los cortos (Q1).')
    print('  Recomendación: explorar max_length=2048 o chunking con agregación de predicciones.')
elif delta_f1 > 0.03:
    print(f'  Los artículos largos (Q4) rinden MEJOR que los cortos (+{delta_f1:.3f} F1).')
    print('  El modelo aprovecha bien el contexto adicional dentro de los 1024 tokens.')
else:
    print(f'  No hay diferencia significativa de rendimiento por longitud (ΔF1={delta_f1:.3f}).')
    print('  El truncado a 1024 tokens no penaliza el rendimiento de forma relevante.')


### Interpretación del análisis de longitud

**Distribución de tokens:** la mayoría de artículos se sitúa entre 500 y 1,500 tokens. La línea roja marca el límite de 1,024 tokens, por encima del cual el texto se trunca.

**Comparativa truncados vs no truncados:** si los artículos truncados (más de 1,024 tokens) y los no truncados muestran accuracy y F1-macro similares, el truncado no introduce un sesgo sistemático. Una diferencia de F1-macro superior a 0.03 justificaría explorar max_length=2048.

**F1-macro por cuartil de longitud:** si Q1 (artículos cortos) y Q4 (artículos largos) obtienen resultados similares, el modelo extrae la señal ideológica de forma robusta independientemente de la longitud. Una caída en Q4 indicaría que el truncado elimina información relevante de los artículos largos.

**Porcentaje truncado vs confianza:** una tendencia negativa (más texto cortado implica menor confianza) indica que el modelo detecta implícitamente la falta de contexto, señal positiva de calibración.

La señal ideológica se concentra habitualmente en el titular y los primeros párrafos de la noticia, por lo que el truncado a 1,024 tokens raramente elimina el contenido más discriminativo.